# Pose predictors

## File Handling
To run predictions a `RobotEnvironment` object and a `HeadsetData` object is needed, those can be loaded from folders or created.

### Creation of RobotEnvironment and HeadsetData
Those 2 datatypes can be created from an GatheredRobotData object and a .vrs file respectively.

In [ ]:
%load_ext autoreload
%autoreload 2
print(__debug__)

import logging
logging.basicConfig(level=logging.INFO)

import seaborn as sns
sns.set_theme(style="whitegrid", context="paper", font_scale=1)


import numpy as np
import random, torch, os, cv2

seed = 1

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
cv2.setRNGSeed(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["PYTHONHASHSEED"] = str(seed)

In [ ]:
from pose_estimation.file_representations.headset_data import HeadsetData, create_robot_bound_headset_data
from pose_estimation.data_interfaces.robot_environment import RobotEnvironment, GatheredRobotData, visualize_robot_camera_environment_combo, XYZImageGenerationConfig, ICPAlignmentConfig


robot_data_folder_location = "../example_datasets/example_small_aruco1"
vrs_file_location = "../example_datasets/small_aruco1_sitting_20fps.vrs"


robot_data = GatheredRobotData.from_folder(robot_data_folder_location)
robot_env = RobotEnvironment.from_gathered_robot_data(
        robot_data = robot_data,
        number_of_sampled_datapoints=10,
        sample_datapoints_based_on_aruco_corectness = False,
        only_sample_robot_datapoints_w_marker_estimates = True,
        markers_use_advanced_removal=True,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=True),
        est3d_xyz_icp_config=ICPAlignmentConfig(do_alginment=False)
)

labeled_headset_data = create_robot_bound_headset_data(
        headset_data = HeadsetData.from_vrs_file(vrs_file_location),
        robot_data = robot_data
)

visualize_loaded_data = True

if visualize_loaded_data:
    visualize_robot_camera_environment_combo(robot_env=robot_env, headset_data=labeled_headset_data)


## Testing Predictors

In [ ]:
from pose_estimation.small_utilities.image_augmentation import Augmentation, Rotate180Deg
from pose_estimation.prediction_on_dataset import PredictionOnDataset
from pose_estimation.predictor_grader import NPredictors1DatasetGrader, GradablePosePredictor
from pose_estimation.pnp.extractors_and_matchers import ExtractAndMatchWrapperConfig, ExtractAndLightGlue
from pose_estimation.pnp.extractors_and_matchers import pose_estimation_ransaac_config_less_precise, pose_estimation_ransaac_config_precise, pose_estimation_ransaac_config_10ms
from pose_estimation.pose_pred_points.pose_pred_points import OnlyPointsPredictor
from pose_estimation.pose_pred_ellipses.pose_pred_points_ellipsoids import *

### Creating Predictors
Now an `PosePredictor` can be created. An `PosePredictor` instance is build upon an `RobotEnvironment` instance and can predict positions from headset-images.

In [ ]:
#pne_optimizer = PyposePNEOptimizer(PyposePnEOptimizerConfig())
#pne_optimizer = PnEDeltaPoseLBFGSOptimizer(time_tracker=tt_pne)

predictor = EllipsoidPredictor(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        cam1_bgr_images=robot_env.robot_bgr_images,
        cam1_xyz_images=robot_env.robot_xyz_images,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            rotation_augmentations=[Rotate180Deg],
            extract_and_match=ExtractAndLightGlue(),
            ransac_config=pose_estimation_ransaac_config_less_precise,
        ),
        pne_optimizer=PnEDeltaPoseAdamOptimizer(),
        ellipsoid_refinement_at_res=(1400, 1400),
        cam1_segmenter=SAM3Segmenter(Sam3Prompt()),
        cam2_segmenter=YOLOv26Segmenter("yoloe-26l-seg.pt"),
        matching_config=GaussianMatchingConfig(dummy_value=0.01),
        visualize_pne_optimisation=False
)

### Grading the Performance of an Initialised Predictor:

In [ ]:
init_predictor_grade = PredictionOnDataset(
    predictor = predictor,
    headset_data = labeled_headset_data,
    number_retry = 1
)
init_predictor_grade.print_summary()


visualize_prediction = True
if visualize_prediction:
    init_predictor_grade.visualize_predictions(
        robot_env=robot_env,
        show_label=False
    )

In [ ]:
# Creation of the Predictors
light_glue = ExtractAndLightGlue()
yolo = YOLOv26Segmenter("yoloe-26l-seg.pt")


no_ellips = GradablePosePredictor(
    creator=OnlyPointsPredictor.get_creation_function(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            rotation_augmentations=[Rotate180Deg],
            extract_and_match=light_glue,
            ransac_config=pose_estimation_ransaac_config_precise,
        )
    ),
    name="LightGlue"
)

ellips_pypose_gd = GradablePosePredictor(
    creator=EllipsoidPredictor.get_creation_function(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            rotation_augmentations=[Rotate180Deg],
            extract_and_match=light_glue,
            ransac_config=pose_estimation_ransaac_config_less_precise,
            display_matching=False
        ),
        pne_optimizer=PyposePNEOptimizer(),
        ellipsoid_refinement_at_res=(1400, 1400),
        cam1_segmenter=SAM3Segmenter(Sam3Prompt()),
        cam2_segmenter=yolo,
        matching_config=GaussianMatchingConfig(dummy_value=0.01),
        ellipsoid_matching_config = PointCloudMatchingConfig(),
        ellipsoid_fitter=SimpleEllipsoidFitterGD(visualize=False, contamination=0.05),
        visualize_matching=False,
        visualize_pne_optimisation=False,
        visualize_segmentation_masks=False,
        visualize_environment_generation = True
    ),
    name="ellips_pypose_gd"
)


ellips_pypose_sf = GradablePosePredictor(
    creator=EllipsoidPredictor.get_creation_function(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            rotation_augmentations=[Rotate180Deg],
            extract_and_match=light_glue,
            ransac_config=pose_estimation_ransaac_config_less_precise,
            display_matching=False
        ),
        pne_optimizer=PyposePNEOptimizer(),
        ellipsoid_refinement_at_res=(1400, 1400),
        cam1_segmenter=SAM3Segmenter(Sam3Prompt()),
        cam2_segmenter=yolo,
        matching_config=GaussianMatchingConfig(dummy_value=0.01),
        ellipsoid_matching_config = PointCloudMatchingConfig(),
        ellipsoid_fitter=SimpleEllipsoidFitter(visualize=False, contamination=0.05),
        visualize_matching=False,
        visualize_pne_optimisation=False,
        visualize_segmentation_masks=False,
        visualize_environment_generation = True
    ),
    name="ellips_pypose_sf"
)

Now those can be used to create a grader object for multiple `PosePredictor` variants.

In [ ]:
# Initialising the grader:
grader = NPredictors1DatasetGrader(
    gradable_pose_predictors=[no_ellips, ellips_pypose_gd ,ellips_pypose_sf],
    headset_data = labeled_headset_data,
    robot_env = robot_env,
)

In [ ]:
from pose_estimation.predictor_grader import *

visualize_trajectories_3d = False
if visualize_trajectories_3d:
    grader.visualize_predictions_3d()

grader.print_summary()

fig1, ax1 = plt.subplots(1, 1, figsize = (16, 8))
grader.plot_time_series_error(ax1, TimeSeriesErrorType.ABS_TRANSLATIONAL)

fig2, ax2 = plt.subplots(1, 1, figsize = (16, 8))
grader.plot_time_series_error(ax2, TimeSeriesErrorType.ABS_ROTATIONAL)

fig3, ax3 = plt.subplots(1, 1, figsize = (16, 8))
grader.plot_creation_times(ax3)

fig4, ax4 = plt.subplots(1, 1, figsize = (8, 5))
grader.plot_successful_frame_prediction_times(ax4)

fig4, ax5 = plt.subplots(1, 1, figsize = (8, 5))
grader.plot_hz_vs_error(ax5, SingleValueErrorType.AVG_GRIPPING_ERROR)


plt.show()

### Visualising the Predictor in Video Format

In [ ]:
from pose_estimation.geometric_utilities.slam2mp4 import VideoGenerator

video_predictor = EllipsoidPredictor(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        cam1_bgr_images=robot_env.robot_bgr_images,
        cam1_xyz_images=robot_env.robot_xyz_images,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            rotation_augmentations=[Rotate180Deg],
            extract_and_match=light_glue,
            ransac_config=pose_estimation_ransaac_config_less_precise,
        ),
        pne_optimizer=PyposePNEOptimizer(),
        ellipsoid_refinement_at_res=(1400, 1400),
        cam1_segmenter=SAM3Segmenter(Sam3Prompt()),
        cam2_segmenter=yolo,
        ellipsoid_fitter=SimpleEllipsoidFitterGD(),
        matching_config=GaussianMatchingConfig(dummy_value=0.01),
        visualize_pne_optimisation=False
)

init_predictor_grade = PredictionOnDataset(
    predictor = video_predictor,
    headset_data = labeled_headset_data,
    number_retry = 1,
    vid_gen=VideoGenerator(fps=20),
    point_cloud=robot_env.robot_xyz_images.reshape(-1,3)
)
init_predictor_grade.print_summary()